# Question 1A — Linear Regression from scratch
**Objective:** predict a continuous numeric output. This notebook implements the model and gradient updates using **NumPy only**; scikit-learn is used for splitting, scaling and evaluation.

Prediction: $\hat{y}_i=\mathbf{x}_i^T\mathbf{w}+b$.

Loss: $J(\mathbf{w},b)=\frac{1}{2m}\sum_{i=1}^{m}(\hat{y}_i-y_i)^2$.

Gradients: $\nabla_{\mathbf{w}}J=\frac{1}{m}X^T(\hat{\mathbf{y}}-\mathbf{y})$, $\partial J/\partial b=\frac{1}{m}\sum_i(\hat y_i-y_i)$.

Updates: $\mathbf{w}\leftarrow\mathbf{w}-\eta\nabla_{\mathbf{w}}J$, $b\leftarrow b-\eta\partial J/\partial b$.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import *
from src.q1_models import gradient_check


## Implementation (not sklearn LinearRegression)
The complete optimization loop and its analytical gradient appear below.

In [2]:
class LinearRegressionGD:
    """Minimize J=(1/(2m)) sum_i (X_i @ w + b - y_i)^2."""
    def __init__(self, learning_rate=0.08, epochs=1200):
        self.learning_rate, self.epochs = learning_rate, epochs
    def _loss_grad(self, X, y, w, b):
        err = X @ w + b - y
        return float(np.mean(err**2)/2), X.T @ err/len(y), float(np.mean(err))
    def fit(self, X, y):
        X, y = np.asarray(X,float), np.asarray(y,float).ravel()
        self.weights_ = np.zeros(X.shape[1]); self.bias_ = 0.
        self.loss_history_ = []
        for epoch in range(self.epochs):
            loss, dw, db = self._loss_grad(X,y,self.weights_,self.bias_)
            self.loss_history_.append(loss)
            self.weights_ -= self.learning_rate*dw
            self.bias_ -= self.learning_rate*db
        return self
    def predict(self,X):
        return np.asarray(X) @ self.weights_ + self.bias_

## Generate data, split 80/20, and standardize using training statistics only

In [3]:
rng=np.random.default_rng(42)
X=rng.normal(size=(360,2))
y=4+3*X[:,0]-2*X[:,1]+rng.normal(0,0.65,len(X))
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
scaler=StandardScaler().fit(X_train)
X_train=scaler.transform(X_train)
X_test=scaler.transform(X_test)
print('Training:',X_train.shape,'Test:',X_test.shape)

Training: (288, 2) Test: (72, 2)


## Fit Gradient Descent and verify the formula numerically

In [4]:
model=LinearRegressionGD(learning_rate=0.08,epochs=1200).fit(X_train,y_train)
check=gradient_check(LinearRegressionGD(),X_train[:20],y_train[:20])
print('Max finite-difference gradient error:',check)
assert check < 1e-6
plt.figure(figsize=(8,4));plt.plot(model.loss_history_);plt.title('Linear training loss');plt.xlabel('Epoch');plt.ylabel('MSE / 2');plt.grid(alpha=.2);plt.show()

Max finite-difference gradient error: 2.226696160789743e-09


/tmp/ipykernel_579/699300012.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.figure(figsize=(8,4));plt.plot(model.loss_history_);plt.title('Linear training loss');plt.xlabel('Epoch');plt.ylabel('MSE / 2');plt.grid(alpha=.2);plt.show()


## Held-out evaluation
MSE and RMSE have squared/original target units; MAE is robust to large errors; R² measures explained variance relative to a mean-prediction baseline.

In [5]:
pred=model.predict(X_test)
print('MSE:',mean_squared_error(y_test,pred))
print('RMSE:',np.sqrt(mean_squared_error(y_test,pred)))
print('MAE:',mean_absolute_error(y_test,pred))
print('R^2:',r2_score(y_test,pred))
plt.figure(figsize=(6,5));plt.scatter(y_test,pred,s=24,alpha=.7)
lims=[min(y_test.min(),pred.min()),max(y_test.max(),pred.max())]
plt.plot(lims,lims,'r--');plt.xlabel('Observed');plt.ylabel('Predicted');plt.title('Held-out predictions');plt.show()

MSE: 0.41759428722908903
RMSE: 0.6462153566954975
MAE: 0.5040398082742661
R^2: 0.9654916280448091


/tmp/ipykernel_579/1126693261.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.plot(lims,lims,'r--');plt.xlabel('Observed');plt.ylabel('Predicted');plt.title('Held-out predictions');plt.show()


**Interpretation:** Because targets were generated from a linear relationship plus random noise, a correctly trained linear model should attain a high R² but cannot predict the noise perfectly. The test set was never used to fit the scaler or model.